# Transfer Evaluation: Zero-Shot Generalization on CBT-Bench (CBT-CD)

### Objective & Purpose
This notebook performs an out-of-domain transfer evaluation to assess how effectively our fine-tuned transformer models generalize beyond our internal training distribution. Specifically, we test the model on the external `Psychotherapy-LLM/CBT-Bench` benchmark without additional fine-tuning.

### Scope of Work
* **Benchmark Acquisition:** Ingest the CBT-CD subset of `Psychotherapy-LLM/CBT-Bench` from Hugging Face (146 items across 10 cognitive distortion categories).
* **Label Harmonization:** Apply an explicit mapping dictionary to translate external benchmark category names into our internal taxonomy (`DISTORTIONS`).
* **Exclusion Audit:** Isolate and count any benchmark samples whose labels cannot be mapped directly to our class definitions, recording the exact number of dropped records.
* **Inference & Metrics:** Evaluate the fine-tuned model against the harmonized test split, generating macro/weighted F1 metrics and a class-by-class classification report to quantify out-of-domain transfer performance.

In [ ]:
# Install the datasets library if it isn't already in your environment
#!pip install datasets transformers pandas scikit-learn torch

In [ ]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

REPO_URL = "https://github.com/lumia-Qcode/EmpowerLens.git"
BRANCH = "izza-space" 

!rm -rf empowerlens && git clone --branch $BRANCH $REPO_URL empowerlens
%cd empowerlens
!pip install -q -r requirements-transformer.txt datasets scikit-learn pandas

In [ ]:
print("Training Phase 1: Original baseline (multi-class) model...")
!python -m src.train_transformer --task multiclass --model mental/mental-roberta-base --seed 42 --device auto --splits data/splits --out checkpoints_mc

print("\nTraining Phase 2: Dedicated (multi-label) model...")
!python -m src.train_transformer --task multilabel --model mental/mental-roberta-base --seed 42 --device auto --splits data/splits --out checkpoints_ml

print("\nBoth models successfully trained and saved locally!")

In [ ]:
from datasets import load_dataset
from src.data import DISTORTIONS
import pandas as pd

print("Acquiring CBT-Bench (CBT-CD split) from Hugging Face...")
hf_dataset = load_dataset("Psychotherapy-LLM/CBT-Bench", "distortions_test", split="train")
df_external = hf_dataset.to_pandas()

CBT_LABEL_CANON = {
    'all-or-nothing thinking': 'all_or_nothing',
    'magnification': 'magnification', 
    'mind reading': 'mind_reading',
    'overgeneralization': 'overgeneralization',
    'personalization': 'personalization',
    'emotional reasoning': 'emotional_reasoning',
    'mental filter': 'mental_filter', 
    'should statements': 'should_statements',
    'labeling': 'labeling',
    'fortune-telling': 'fortune_telling'
}

def strict_canon_map(label_array):
    mapped = []
    for label in label_array:
        key = str(label).strip()
        if key not in CBT_LABEL_CANON:
            raise ValueError(f"Unmapped external label {label!r}. No fuzzy matching allowed.")
        mapped.append(CBT_LABEL_CANON[key])
    return mapped

df_external['internal_labels'] = df_external['distortions'].apply(strict_canon_map)

original_count = len(df_external)
df_filtered = df_external[df_external['internal_labels'].map(len) > 0].copy()
dropped_count = original_count - len(df_filtered)

print(f"Total loaded: {original_count}")
print(f"Successfully mapped: {len(df_filtered)}")
print(f"Dropped items (exclusion audit): {dropped_count}")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import classification_report
import numpy as np
import os

device = "cuda" if torch.cuda.is_available() else "cpu"
texts = df_filtered["ori_text"].tolist()
true_labels_arrays = df_filtered["internal_labels"].tolist()

mlb = MultiLabelBinarizer(classes=DISTORTIONS)
true_matrix = mlb.fit_transform(true_labels_arrays)
batch_size = 16

# ---------------------------------------------------------
# 1. RUN MULTI-CLASS INFERENCE (Argmax)
# ---------------------------------------------------------
print(f"Loading Phase 1 (Multi-Class) model onto {device}...")
MC_PATH = "checkpoints_mc/multiclass_mental-roberta-base_42"
tokenizer_mc = AutoTokenizer.from_pretrained(MC_PATH)
model_mc = AutoModelForSequenceClassification.from_pretrained(MC_PATH).to(device)
model_mc.eval()

preds_mc_list = []
with torch.no_grad():
    for i in range(0, len(texts), batch_size):
        inputs = tokenizer_mc(texts[i:i+batch_size], padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
        logits = model_mc(**inputs).logits
        batch_preds = torch.argmax(logits, dim=1).cpu().numpy()
        
        batch_ml_format = np.zeros((len(batch_preds), len(DISTORTIONS)), dtype=int)
        for j, pred_idx in enumerate(batch_preds):
            if pred_idx > 0: 
                batch_ml_format[j, pred_idx - 1] = 1
        preds_mc_list.extend(batch_ml_format)

report_mc = classification_report(true_matrix, np.array(preds_mc_list), target_names=DISTORTIONS, zero_division=0)

# ---------------------------------------------------------
# 2. RUN MULTI-LABEL INFERENCE (Sigmoid Thresholding)
# ---------------------------------------------------------
print(f"Loading Phase 2 (Multi-Label) model onto {device}...")
ML_PATH = "checkpoints_ml/multilabel_mental-roberta-base_42"
tokenizer_ml = AutoTokenizer.from_pretrained(ML_PATH)
model_ml = AutoModelForSequenceClassification.from_pretrained(ML_PATH, problem_type="multi_label_classification").to(device)
model_ml.eval()

preds_ml_list = []
with torch.no_grad():
    for i in range(0, len(texts), batch_size):
        inputs = tokenizer_ml(texts[i:i+batch_size], padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
        logits = model_ml(**inputs).logits
        
        # Multi-label logic: Apply Sigmoid and 0.5 threshold to score each column independently
        probs = torch.sigmoid(logits).cpu().numpy()
        batch_preds = (probs > 0.5).astype(int)
        preds_ml_list.extend(batch_preds)

report_ml = classification_report(true_matrix, np.array(preds_ml_list), target_names=DISTORTIONS, zero_division=0)

# ---------------------------------------------------------
# 3. COMPILE AND EXPORT THE FINAL COMBINED REPORT
# ---------------------------------------------------------
output_dir = "results_transfer"
os.makedirs(output_dir, exist_ok=True)
report_path = os.path.join(output_dir, "combined_transfer_evaluation.txt")

explanation_text = (
    "=== WHY WE RECORDED BOTH SCORES ===\n"
    "The original research paper formulated cognitive distortion detection strictly as a multi-class classification problem. "
    "We recorded the multi-class scores first to establish a faithful baseline that accurately replicates those original constraints. "
    "We then recorded the multi-label scores to demonstrate empirically how upgrading to a dedicated multi-label architecture "
    "resolves structural bottlenecks and correctly handles the overlapping, real-world data found in CBT-Bench.\n\n"
)

with open(report_path, "w") as f:
    f.write("=== CBT-BENCH COMBINED TRANSFER EVALUATION ===\n")
    f.write(f"Items successfully mapped: {len(df_filtered)}\n")
    f.write(f"Items dropped (exclusion audit): {dropped_count}\n\n")
    f.write(explanation_text)
    f.write("--- PHASE 1: MULTI-CLASS BASELINE RESULTS ---\n")
    f.write(report_mc)
    f.write("\n--- PHASE 2: MULTI-LABEL ARCHITECTURE RESULTS ---\n")
    f.write(report_ml)

print(f"\nCombined report successfully saved to {report_path}")

In [ ]:
!zip -r results_transfer.zip results_transfer/
from IPython.display import FileLink
display(FileLink('results_transfer.zip'))